In [4]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import shutil
import sys

project_root = Path.cwd().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Set device, automatically uses GPU if available, falls back to CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(torch.cuda.is_available())        # True if NVIDIA GPU detected

project_root = Path.cwd().resolve().parent.parent
processed_dir = project_root / 'src' / 'data' / 'processed'

from src.models.model import PitStrategyNet, PitStrategyGRUAttention, PitStrategyLSTM

# Load data
X_train = np.load(processed_dir / 'X_train.npy')
y_train = np.load(processed_dir / 'y_train.npy')
X_val = np.load(processed_dir / 'X_val.npy')
y_val = np.load(processed_dir / 'y_val.npy')

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

Using device: cpu
False
X_train: (1519, 3, 37), y_train: (1519,)


In [5]:
import yaml
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

project_root = Path.cwd().resolve().parent.parent
with open(project_root / 'src' / 'config.yaml') as f:
    config = yaml.safe_load(f)

# Load the model in
model = PitStrategyNet(
    input_dim=config['model']['input_dim'],
    hidden_dim_1=config['model']['hidden_dim_1'],
    hidden_dim_2=config['model']['hidden_dim_2'],
    dropout_rate=config['model']['dropout_rate'],
    num_classes=config['model']['num_classes']
)

'''model = PitStrategyGRUAttention(
    input_dim=config['model']['input_dim'],
    hidden_dim=config['model']['hidden_dim'],
    num_layers=config['model']['num_layers'],
    dropout_rate=config['model']['dropout_rate'],
    num_classes=config['model']['num_classes']
)'''

'''model = PitStrategyLSTM(
    input_dim=config['model']['input_dim'],
    hidden_dim=config['model']['hidden_dim'],
    num_layers=config['model']['num_layers'],
    dropout_rate=config['model']['dropout_rate'],
    num_classes=config['model']['num_classes']
)'''

classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr = config['training']['learning_rate'],
    weight_decay = 1e-4  # L2 regularisation, penalises large weights
)

# This reduces the learning rate when the loss stops improving
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode = 'min',
    factor = 0.25,
    patience = 100,
)

num_epochs = config['training']['epochs']
batch_size = config['training']['batch_size']

AttributeError: module 'torch' has no attribute '_utils'

In [ ]:
# Loads data in a specific format to the model
class LapDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)  # long for CrossEntropyLoss

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# DataLoader wraps the dataset for easier process and hands the model in batch sizes
train_loader = DataLoader(LapDataset(X_train, y_train), batch_size=batch_size, shuffle=False, pin_memory=False, num_workers=0)
val_loader = DataLoader(LapDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

In [ ]:
models_dir = project_root / 'src' / 'models' / config['paths']['folder_name']
models_dir.mkdir(exist_ok=True)

best_val_loss = float('inf')
patience_ctr = 0
patience = 50

for epoch in range(num_epochs):
    # Training
    model.train()
    train_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss  = criterion(outputs, y_batch)
            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss  = val_loss   / len(val_loader)
    scheduler.step(avg_val_loss)  # scheduler now uses val loss

    # Early stopping on val loss
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_ctr  = 0
        torch.save({
            'model_state_dict': model.state_dict(),
            'input_dim': config['model']['input_dim'],
        }, models_dir / config['paths']['model_filename'])
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Val Loss: {avg_val_loss:.4f} | "
              f"LR: {current_lr:.6f} | "
              f"Best Val: {best_val_loss:.4f}")


In [ ]:
# Validation accuracy
model.eval()
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)

val_preds = []
val_labels = []

with torch.no_grad():
    for i in range(0, len(X_val_tensor), 32):
        batch = X_val_tensor[i:i+32]
        outputs = model(batch)
        preds = torch.argmax(outputs, dim=1)
        val_preds.append(preds.cpu())
        val_labels.append(torch.tensor(y_val[i:i+32]))

val_preds = torch.cat(val_preds)
val_labels = torch.cat(val_labels)

val_accuracy = (val_preds == val_labels).float().mean().item()
print(f"Validation accuracy: {val_accuracy:.4f}")

In [ ]:
# Save weights
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': config['model']['input_dim'],
}, models_dir / config['paths']['model_filename'])

shutil.copy(project_root / 'src' / 'config.yaml',
            models_dir / config['paths']['config_name'])

print("Model saved.")